# Colab Gradio App: ViT + MedGemma Base + MedGemma QLoRA

This notebook is a deployment/demo notebook. It loads:

1. a trained **ViT image classifier**,
2. **MedGemma without QLoRA adapters**,
3. **MedGemma with QLoRA adapters**, then
4. launches a **Gradio app** that compares all three outputs for an uploaded histology patch.

Before running: set Colab runtime to **GPU**, add your Hugging Face token as a Colab secret named `HF_TOKEN`, and update the model paths in the configuration cell.


## 1. Install dependencies


In [ ]:
!pip install -q -U torch torchvision torchaudio
!pip install -q -U transformers accelerate bitsandbytes peft gradio pandas sentencepiece protobuf huggingface_hub
!pip install -q Pillow==9.5.0

## 2. Imports, Drive mount, and Hugging Face login


In [ ]:
import os
import gc
import glob
import torch
import numpy as np
import pandas as pd
import gradio as gr
from PIL import Image

from google.colab import drive, userdata
from huggingface_hub import login
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)
from peft import PeftModel

# Mount Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# Optional Colab bitsandbytes CUDA helper
os.environ.setdefault('BNB_CUDA_VERSION', '124')
nvidia_libs = glob.glob('/usr/local/lib/python3.*/dist-packages/nvidia/*/lib')
if nvidia_libs:
    os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + ':' + ':'.join(nvidia_libs)

# Login for gated MedGemma access
try:
    login(token=userdata.get('HF_TOKEN'))
    print('Hugging Face login successful.')
except Exception as e:
    print('Hugging Face login failed. Add HF_TOKEN in Colab Secrets and accept MedGemma access.')
    print(e)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print(torch.cuda.get_device_name(0))


Mounted at /content/drive
Hugging Face login successful.
Device: cuda
Tesla T4


## 3. Configuration

Edit these values to match your Drive folder and saved models.


In [ ]:
# ===== EDIT THESE PATHS =====
PROJECT_DIR = '/content/drive/MyDrive/MedGemma_Colorectal_Project_v2'

# ViT classifier folder created by save_pretrained / Trainer.save_model
VIT_MODEL_DIR = f'{PROJECT_DIR}/models/vit-baseline-v2'

# Base gated MedGemma model on Hugging Face
MEDGEMMA_MODEL_ID = 'google/medgemma-1.5-4b-it'

# QLoRA adapter folder created by trainer.save_model and processor.save_pretrained
MEDGEMMA_QLORA_ADAPTER_DIR = f'{PROJECT_DIR}/models/medgemma-crc-qlora-v2'

# Labels must match ViT training order
LABEL_NAMES = [
    'ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM'
]

LABEL_DESCRIPTIONS = {
    'ADI': 'adipose tissue',
    'BACK': 'background',
    'DEB': 'debris',
    'LYM': 'lymphocytes',
    'MUC': 'mucus',
    'MUS': 'smooth muscle',
    'NORM': 'normal colon mucosa',
    'STR': 'cancer-associated stroma',
    'TUM': 'colorectal adenocarcinoma epithelium',
}

CLASS_PROMPT = f"""
You are assisting with colorectal histology patch classification.
Classify the image into exactly one of these labels:
{', '.join(LABEL_NAMES)}.
Meanings: {LABEL_DESCRIPTIONS}.
Return only the label code and a one-sentence rationale.
""".strip()

print('ViT path:', VIT_MODEL_DIR)
print('QLoRA adapter path:', MEDGEMMA_QLORA_ADAPTER_DIR)


ViT path: /content/drive/MyDrive/MedGemma_Colorectal_Project_v2/models/vit-baseline-v2
QLoRA adapter path: /content/drive/MyDrive/MedGemma_Colorectal_Project_v2/models/medgemma-crc-qlora-v2


## 4. Load ViT classifier


In [ ]:
vit_processor = AutoImageProcessor.from_pretrained(VIT_MODEL_DIR, use_fast=True)
vit_model = AutoModelForImageClassification.from_pretrained(VIT_MODEL_DIR).to(device)
vit_model.eval()
print('ViT loaded.')


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViT loaded.


## 5. Load base MedGemma without QLoRA

This uses 4-bit loading for memory efficiency, but no LoRA adapter is attached.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_med_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)
base_med_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
base_med_model.eval()
print('Base MedGemma loaded without QLoRA.')


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Base MedGemma loaded without QLoRA.


## 6. Load MedGemma with QLoRA adapter

This loads a separate base MedGemma instance and attaches the saved QLoRA adapter.


In [ ]:
qlora_med_processor = AutoProcessor.from_pretrained(MEDGEMMA_QLORA_ADAPTER_DIR)
qlora_base_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
qlora_med_model = PeftModel.from_pretrained(qlora_base_model, MEDGEMMA_QLORA_ADAPTER_DIR)
qlora_med_model.eval()
print('MedGemma with QLoRA adapter loaded.')


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

MedGemma with QLoRA adapter loaded.


## 7. Prediction helpers


In [ ]:
def normalize_image(image):
    if image is None:
        return None
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    return image.convert('RGB')


def predict_vit(image):
    image = normalize_image(image)
    inputs = vit_processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = vit_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0].detach().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    rows = []
    for i, p in enumerate(probs):
        label = LABEL_NAMES[i] if i < len(LABEL_NAMES) else str(i)
        rows.append({'Label': label, 'Meaning': LABEL_DESCRIPTIONS.get(label, ''), 'Confidence': float(p)})
    scores_df = pd.DataFrame(rows).sort_values('Confidence', ascending=False)
    return LABEL_NAMES[pred_idx], float(probs[pred_idx]), scores_df


def _predict_medgemma(image, model, processor, max_new_tokens=80):
    image = normalize_image(image)
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': CLASS_PROMPT},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, images=image, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    # Keep the generated answer readable if the prompt is echoed.
    if CLASS_PROMPT[:40] in decoded:
        decoded = decoded.split(CLASS_PROMPT[:40])[-1]
    return decoded.strip()


def predict_base_medgemma(image):
    return _predict_medgemma(image, base_med_model, base_med_processor)


def predict_qlora_medgemma(image):
    return _predict_medgemma(image, qlora_med_model, qlora_med_processor)


## 8. Gradio app


In [ ]:
def demo_predict(image):
    if image is None:
        empty_df = pd.DataFrame(columns=['Label', 'Meaning', 'Confidence'])
        return '', 0.0, empty_df, '', '', 'Upload a histology patch to run all three models.'

    vit_label, vit_conf, vit_scores = predict_vit(image)
    base_text = predict_base_medgemma(image)
    qlora_text = predict_qlora_medgemma(image)

    summary = (
        f"ViT predicts **{vit_label}** ({LABEL_DESCRIPTIONS.get(vit_label, '')}) "
        f"with {vit_conf:.2%} confidence. Compare the base MedGemma response with the QLoRA-adapted response below."
    )
    return vit_label, round(vit_conf, 4), vit_scores, base_text, qlora_text, summary

with gr.Blocks(title='ViT vs MedGemma vs MedGemma QLoRA') as demo:
    gr.Markdown('# Clinical Histology Demo: ViT + MedGemma Base + MedGemma QLoRA')
    gr.Markdown('Upload a colorectal histology patch. This demo compares a ViT classifier, base MedGemma, and QLoRA-adapted MedGemma.')

    with gr.Row():
        image_input = gr.Image(type='pil', label='Upload histology patch')
        with gr.Column():
            vit_label = gr.Textbox(label='ViT predicted label')
            vit_conf = gr.Number(label='ViT confidence')
            summary = gr.Markdown(label='Summary')

    vit_scores = gr.Dataframe(label='ViT class confidence scores')
    base_output = gr.Textbox(label='Base MedGemma output: no QLoRA', lines=5)
    qlora_output = gr.Textbox(label='MedGemma output: with QLoRA adapter', lines=5)

    run_btn = gr.Button('Run all models')
    run_btn.click(
        fn=demo_predict,
        inputs=image_input,
        outputs=[vit_label, vit_conf, vit_scores, base_output, qlora_output, summary],
    )

print('Gradio app is defined. Run the next cell to launch it.')


Gradio app is defined. Run the next cell to launch it.


## 9. Launch the app


In [ ]:
demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9ad50cd1725d621869.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


## 10. Optional memory cleanup

Run this only after closing the app or when you need to reload models.


In [ ]:
# Optional cleanup
# demo.close()
# del vit_model, base_med_model, qlora_med_model, qlora_base_model
# gc.collect()
# torch.cuda.empty_cache()
